In [1]:
import os, glob, subprocess, math, logging
import numpy as np
import pandas as pd
import xarray as xr
import georinex as gr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [5]:
"""
GNSS → PWV Pipeline  (CUSV Bangkok)
=====================================
Step 0  : Config & paths
Step 1  : Decompress .gz → RINEX + merge blocks
Step 2  : Quality check ด้วย georinex
Step 3  : ดาวน์โหลด SP3 + CLK (precise ephemeris)
Step 4  : PPP processing ด้วย RTKLIB → ZTD
Step 5  : ดาวน์โหลด ERA5 surface Ps, Ts
Step 6  : คำนวณ ZHD → ZWD → Tm → PWV
Step 7  : Plot PWV time series + dPWV/dt

Install:
    pip install georinex cdsapi xarray numpy pandas matplotlib
    brew install rtklib        # macOS
"""

import os, glob, subprocess, math, logging
import numpy as np
import pandas as pd
import xarray as xr
import georinex as gr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger(__name__)


# ══════════════════════════════════════════════════════════════
# STEP 0 — CONFIG  (แก้ไขเฉพาะส่วนนี้)
# ══════════════════════════════════════════════════════════════

# BASE_DIR = "/Users/toyzaa/Desktop/th_rain"
RAW_DIR  = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw"                          # ไฟล์ .gz ที่ดาวน์โหลดมา
PROC_DIR = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_proc"            # ★ RINEX หลัง decompress (Step 1 output / Step 2+ input)
EPH_DIR  = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/ephemeris"                         # SP3 + CLK files
ZTD_DIR  = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/ztd_output"                        # RTKLIB .pos output
ERA5_DIR = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/era5"                              # ERA5 NetCDF
FIG_DIR  = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/figures"                           # PNG output

# ── ตรวจสอบ SSD mount ก่อนดำเนินการ ─────────────────────────
_ssd_root = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS"
if not os.path.exists(_ssd_root):
    raise RuntimeError(
        f"ไม่พบ SSD ที่ mount อยู่:\n  {_ssd_root}\n"
        "ตรวจสอบว่าเสียบ drive และ mount แล้ว"
    )

# สร้าง folder อัตโนมัติถ้ายังไม่มี
for _d in [RAW_DIR, PROC_DIR, EPH_DIR, ZTD_DIR, ERA5_DIR, FIG_DIR]:
    os.makedirs(_d, exist_ok=True)

# Station metadata — CUSV Chulalongkorn University, Bangkok
STATION = "cusv"
LAT_DEG = 13.7308    # latitude  (°N)
LON_DEG = 100.5322   # longitude (°E)
H_M     = 35.0       # ellipsoidal height (m)

# Observation date ← ตรงกับไฟล์ cusv2281200.24o.gz
OBS_YEAR = 2024
OBS_DOY  = 228       # day-of-year  (228 = 15 Aug 2024)
OBS_HOUR = 12        # UTC hour ที่เริ่มต้น

# ── Demo flags (เปลี่ยนเป็น False เมื่อมี tool จริง) ──────────
USE_DEMO_ZTD  = True   # True = ใช้ ZTD จำลอง  | False = รัน RTKLIB จริง
USE_DEMO_ERA5 = True   # True = ใช้ ERA5 จำลอง | False = ดาวน์โหลดจาก CDS API


# ══════════════════════════════════════════════════════════════
# STEP 1 — DECOMPRESS .gz  →  RINEX  +  MERGE BLOCKS
# ══════════════════════════════════════════════════════════════

def decompress_all() -> list[str]:
    """
    แตกไฟล์ทุก .gz ใน RAW_DIR → PROC_DIR
    รองรับทั้ง RINEX 2 (.24o) และ RINEX 3 (.rnx)
    Returns: sorted list ของ RINEX paths ที่พร้อมใช้งาน
    """
    gz_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.gz")))
    if not gz_files:
        raise FileNotFoundError(
            f"ไม่พบไฟล์ .gz ใน:\n  {RAW_DIR}\n"
            "ตรวจสอบว่าดาวน์โหลดไฟล์มาไว้ที่ path นี้แล้ว"
        )

    log.info(f"พบ {len(gz_files)} ไฟล์ .gz ใน {RAW_DIR}")
    rinex_paths = []

    for gz in gz_files:
        basename = os.path.basename(gz).replace(".gz", "")
        out_path = os.path.join(PROC_DIR, basename)

        if os.path.exists(out_path):
            log.info(f"  ข้าม (มีอยู่แล้ว): {basename}")
        else:
            with open(out_path, "wb") as out_f:
                result = subprocess.run(
                    ["gunzip", "-c", gz],
                    stdout=out_f,
                    stderr=subprocess.PIPE,
                )
            if result.returncode != 0:
                log.error(
                    f"  gunzip ล้มเหลว: {basename}\n"
                    f"  {result.stderr.decode().strip()}"
                )
                if os.path.exists(out_path):
                    os.remove(out_path)
                continue
            log.info(f"  ✅ แตกไฟล์สำเร็จ: {basename}  →  {PROC_DIR}")

        rinex_paths.append(out_path)

    if not rinex_paths:
        raise RuntimeError("ไม่มีไฟล์ RINEX ที่แตกได้เลย — ตรวจสอบไฟล์ .gz ใหม่")

    log.info(f"รวม {len(rinex_paths)} RINEX blocks พร้อมใช้งานที่ {PROC_DIR}")
    return rinex_paths


def merge_rinex_blocks(rinex_paths: list[str]) -> xr.Dataset:
    """
    โหลดและรวม RINEX blocks ทั้งหมดจาก PROC_DIR (SSD) เป็น xarray.Dataset เดียว
    4 blocks × 15 นาที = 1 ชั่วโมงของข้อมูล 1 Hz (3,600 epochs)
    อ่านจาก: /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_proc
    Returns: xr.Dataset sorted by time
    """
    log.info(f"กำลังโหลด {len(rinex_paths)} blocks จาก SSD:")
    log.info(f"  {PROC_DIR}")
    datasets = []

    for path in rinex_paths:
        try:
            ds = gr.load(path, use=["C1C", "L1C", "L2C", "S1C", "S2C"])
            datasets.append(ds)
            log.info(
                f"  {os.path.basename(path)} | "
                f"epochs={len(ds.time):,} | "
                f"sats={len(ds.sv)}"
            )
        except Exception as e:
            log.warning(f"  อ่านไม่ได้: {os.path.basename(path)} — {e}")

    if not datasets:
        raise RuntimeError("ไม่สามารถอ่าน RINEX block ใดได้เลย")

    combined = xr.concat(datasets, dim="time").sortby("time")
    t0 = str(combined.time.values[0])[:19]
    t1 = str(combined.time.values[-1])[:19]
    log.info(f"Merged: {len(combined.time):,} epochs | {t0} → {t1}")
    return combined


# ══════════════════════════════════════════════════════════════
# STEP 2 — QUALITY CHECK
# ══════════════════════════════════════════════════════════════

def quality_check(obs_ds: xr.Dataset) -> dict:
    """
    ตรวจคุณภาพ RINEX dataset ที่อ่านมาจาก SSD
      path: /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_proc
    ตรวจ: SNR, จำนวน satellites, completeness, multipath proxy
    Returns: dict สรุป QC metrics
    """
    qc = {
        "n_epochs" : int(len(obs_ds.time)),
        "n_sats"   : int(len(obs_ds.sv)),
        "t_start"  : str(obs_ds.time.values[0])[:19],
        "t_end"    : str(obs_ds.time.values[-1])[:19],
    }

    # ── SNR L1 (S1C) ──────────────────────────────────────────
    if "S1C" in obs_ds:
        snr = obs_ds["S1C"].values.flatten()
        snr = snr[~np.isnan(snr)]
        qc["snr_mean_dBHz"]   = round(float(snr.mean()), 1)
        qc["snr_min_dBHz"]    = round(float(snr.min()),  1)
        qc["snr_pct_below30"] = round(float((snr < 30).mean() * 100), 1)
        status = "⚠️  ต่ำ" if qc["snr_mean_dBHz"] < 30 else "✅ ดี"
        log.info(f"  SNR L1: {qc['snr_mean_dBHz']} dBHz mean  {status}")

    # ── Multipath proxy ────────────────────────────────────────
    if "C1C" in obs_ds and "L1C" in obs_ds:
        mp = (obs_ds["C1C"] - obs_ds["L1C"]).values.flatten()
        mp = mp[~np.isnan(mp)]
        if len(mp) > 0:
            qc["multipath_proxy_m"] = round(float(np.std(mp)), 3)
            log.info(f"  Multipath proxy σ: {qc['multipath_proxy_m']} m")

    # ── Data completeness (epochs ที่มี ≥ 4 sats) ─────────────
    if "C1C" in obs_ds:
        n_valid = (~np.isnan(obs_ds["C1C"].values)).sum(axis=1)
        pct     = float((n_valid >= 4).mean() * 100)
        qc["completeness_pct"] = round(pct, 1)
        log.info(f"  Completeness (≥4 sats): {pct:.1f}%")

    log.info("── QC Summary ──")
    for k, v in qc.items():
        log.info(f"  {k}: {v}")
    return qc


# ══════════════════════════════════════════════════════════════
# STEP 3 — ดาวน์โหลด SP3 + CLK (Precise Ephemeris)
# ══════════════════════════════════════════════════════════════

def doy_to_gpsweek(year: int, doy: int) -> tuple[int, int]:
    """แปลง year + doy → (GPS week, day-of-week)"""
    from datetime import date, timedelta
    d          = date(year, 1, 1) + timedelta(days=doy - 1)
    gps_epoch  = date(1980, 1, 6)
    total_days = (d - gps_epoch).days
    return total_days // 7, total_days % 7


def download_ephemeris() -> dict[str, str | None]:
    """
    ดาวน์โหลด IGS Final SP3 + CLK จาก NASA CDDIS → EPH_DIR
    หมายเหตุ: IGS Final products พร้อมใช้หลัง ~13 วัน
    Returns: {"sp3": path_or_None, "clk": path_or_None}
    """
    import urllib.request

    gps_week, gps_dow = doy_to_gpsweek(OBS_YEAR, OBS_DOY)
    log.info(f"GPS week={gps_week}, dow={gps_dow}  →  EPH_DIR: {EPH_DIR}")

    base  = "https://cddis.nasa.gov/archive/gnss/products"
    files = {
        "sp3": f"igs{gps_week}{gps_dow}.sp3.Z",
        "clk": f"igs{gps_week}{gps_dow}.clk_30s.Z",
    }

    paths: dict[str, str | None] = {}
    for ftype, fname in files.items():
        final_path = os.path.join(EPH_DIR, fname.replace(".Z", ""))

        if os.path.exists(final_path):
            log.info(f"  มีอยู่แล้ว: {os.path.basename(final_path)}")
            paths[ftype] = final_path
            continue

        url      = f"{base}/{gps_week}/{fname}"
        tmp_path = os.path.join(EPH_DIR, fname)

        log.info(f"  ดาวน์โหลด {ftype.upper()}: {url}")
        try:
            urllib.request.urlretrieve(url, tmp_path)
            subprocess.run(["uncompress", tmp_path], check=True)
            paths[ftype] = final_path
            log.info(f"  ✅ {ftype.upper()} พร้อมที่: {final_path}")
        except Exception as e:
            log.error(f"  ❌ ล้มเหลว: {fname} — {e}")
            paths[ftype] = None

    return paths


# ══════════════════════════════════════════════════════════════
# STEP 4 — PPP PROCESSING (RTKLIB) → ZTD time series
# ══════════════════════════════════════════════════════════════

RTKLIB_CONF = """\
pos1-posmode    =ppp-static
pos1-frequency  =l1+l2
pos1-tropopt    =est-ztd
pos1-ionoopt    =dual-freq
pos1-sateph     =prec
pos1-navsys     =1+2+4+8
pos1-elmask     =15
pos2-armode     =off
out-outstat     =residual
out-solstat     =0
"""


def run_ppp(rinex_obs: str, rinex_nav: str,
            sp3_path: str, clk_path: str) -> str:
    """
    รัน RTKLIB rnx2rtkp ใน PPP-static mode → ZTD
    Output .pos file บันทึกที่ ZTD_DIR
    Returns: path ของ .pos output file
    """
    conf_path = os.path.join(ZTD_DIR, "ppp.conf")
    out_path  = os.path.join(ZTD_DIR, f"{STATION}_ztd.pos")

    with open(conf_path, "w") as f:
        f.write(RTKLIB_CONF)

    for label, path in [("obs", rinex_obs), ("nav", rinex_nav),
                         ("sp3", sp3_path),  ("clk", clk_path)]:
        if not path or not os.path.exists(path):
            raise FileNotFoundError(f"ไม่พบ {label} file: {path}")

    cmd = ["rnx2rtkp", "-k", conf_path, "-x", "4",
           "-o", out_path,
           rinex_obs, rinex_nav, sp3_path, clk_path]

    log.info(f"รัน RTKLIB PPP:\n  {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        log.error(f"RTKLIB error:\n{result.stderr[-800:]}")
    else:
        log.info(f"✅ PPP เสร็จ → {out_path}")
    return out_path


def parse_ztd(pos_file: str) -> pd.DataFrame:
    """
    อ่าน ZTD จาก RTKLIB .pos output (debug level 4)
    Column order: date time lat lon h Q ns sdn sde sdu sdne sdeu sdun age ZTD
    Returns: DataFrame index=time, columns=[ZTD_m]
    """
    if not os.path.exists(pos_file):
        raise FileNotFoundError(f"ไม่พบ pos file: {pos_file}")

    rows = []
    with open(pos_file) as f:
        for line in f:
            if line.startswith("%") or not line.strip():
                continue
            parts = line.split()
            if len(parts) < 15:
                continue
            try:
                epoch = pd.Timestamp(f"{parts[0]} {parts[1]}")
                ztd   = float(parts[14])    # ZTD (m) — column 14
                rows.append({"time": epoch, "ZTD_m": ztd})
            except (ValueError, IndexError):
                continue

    if not rows:
        raise RuntimeError(f"ไม่พบ ZTD data ใน {pos_file}")

    df = pd.DataFrame(rows).set_index("time").sort_index().dropna()
    log.info(f"ZTD: {len(df)} epochs | {df.ZTD_m.min():.4f}–{df.ZTD_m.max():.4f} m")
    return df


# ══════════════════════════════════════════════════════════════
# STEP 5 — ERA5 Surface Ps, Ts
# ══════════════════════════════════════════════════════════════

def download_era5() -> str:
    """
    ดาวน์โหลด ERA5 single-level (surface_pressure + 2m_temperature) ทั้งวัน
    บันทึกที่ ERA5_DIR
    ต้องตั้ง ~/.cdsapirc ก่อน: https://cds.climate.copernicus.eu/api-how-to
    Returns: path ของ NetCDF file
    """
    import cdsapi
    from datetime import date, timedelta

    d       = date(OBS_YEAR, 1, 1) + timedelta(days=OBS_DOY - 1)
    datestr = d.strftime("%Y-%m-%d")
    outpath = os.path.join(ERA5_DIR, f"era5_sfc_{datestr}.nc")

    if os.path.exists(outpath):
        log.info(f"ERA5 มีอยู่แล้วที่: {outpath}")
        return outpath

    area = [LAT_DEG + 1, LON_DEG - 1, LAT_DEG - 1, LON_DEG + 1]   # [N,W,S,E]
    log.info(f"ดาวน์โหลด ERA5: {datestr}  area={area}  →  {ERA5_DIR}")

    c = cdsapi.Client()
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable":     ["surface_pressure", "2m_temperature"],
            "year":         str(OBS_YEAR),
            "month":        f"{d.month:02d}",
            "day":          f"{d.day:02d}",
            "time":         [f"{h:02d}:00" for h in range(24)],
            "area":         area,
            "format":       "netcdf",
        },
        outpath,
    )
    log.info(f"✅ ERA5 ดาวน์โหลดเสร็จ → {outpath}")
    return outpath


def extract_era5_surface(nc_path: str) -> pd.DataFrame:
    """
    Extract Ps (hPa) และ Ts (K) ที่จุดใกล้ station ที่สุดจาก ERA5 NetCDF
    Returns: DataFrame index=time, columns=[Ps_hPa, Ts_K]
    """
    ds     = xr.open_dataset(nc_path)
    pt     = ds.sel(latitude=LAT_DEG, longitude=LON_DEG, method="nearest")
    Ps_hPa = pt["sp"].values  / 100.0    # Pa → hPa
    Ts_K   = pt["t2m"].values            # K
    times  = pd.to_datetime(pt["time"].values)

    df = pd.DataFrame({"Ps_hPa": Ps_hPa, "Ts_K": Ts_K}, index=times)
    log.info(
        f"ERA5 | Ps={Ps_hPa.mean():.1f}±{Ps_hPa.std():.1f} hPa | "
        f"Ts={Ts_K.mean():.1f}±{Ts_K.std():.1f} K"
    )
    return df


# ══════════════════════════════════════════════════════════════
# STEP 6 — ZTD → ZHD → ZWD → Tm → Pi → PWV
# ══════════════════════════════════════════════════════════════

def compute_pwv(ztd_m: float, Ps_hPa: float, Ts_K: float) -> dict:
    """
    คำนวณ PWV จาก ZTD + surface meteorological parameters

    Chain:
        ZHD  = 0.0022768 × Ps / f(φ, H)     [Saastamoinen 1972]
        ZWD  = ZTD − ZHD
        Tm   = 70.2 + 0.72 × Ts             [Bevis et al. 1992]
        Π    = 1e6 / [Rv × (k3/Tm + k2')]
        PWV  = Π × ZWD  (mm)

    Constants [Bevis 1994]:
        k2'  = 22.97   K/hPa
        k3   = 377600  K²/hPa
        Rv   = 461.495 J/(kg·K)
    """
    # 1. ZHD (Saastamoinen 1972)
    phi   = math.radians(LAT_DEG)
    f_lat = 1.0 - 0.00266 * math.cos(2.0 * phi) - 0.00028e-3 * H_M
    ZHD   = 0.0022768 * Ps_hPa / f_lat           # metres

    # 2. ZWD
    ZWD = ztd_m - ZHD                             # metres
    if ZWD < 0:
        log.warning(
            f"ZWD < 0 ({ZWD:.5f} m) — "
            f"ตรวจสอบ Ps={Ps_hPa:.1f} hPa / ZTD={ztd_m:.4f} m"
        )

    # 3. Weighted mean temperature (Bevis 1992 linear approx)
    Tm = 70.2 + 0.72 * Ts_K                       # K

    # 4. Pi conversion factor
    Rv  = 461.495    # J/(kg·K)
    k2p = 22.97      # K/hPa
    k3  = 377600.0   # K²/hPa
    Pi  = 1e6 / (Rv * (k3 / Tm + k2p))

    # 5. PWV
    PWV = Pi * ZWD * 1000.0                        # m → mm

    return {
        "ZHD_m":  round(ZHD,  5),
        "ZWD_m":  round(ZWD,  5),
        "Tm_K":   round(Tm,   2),
        "Pi":     round(Pi,   5),
        "PWV_mm": round(PWV,  2),
    }


def compute_pwv_timeseries(ztd_df: pd.DataFrame,
                            era5_df: pd.DataFrame) -> pd.DataFrame:
    """
    คำนวณ PWV ทุก epoch + dPWV/dt (rainbomb precursor indicator)
    ERA5 (hourly) จะถูก interpolate ไปยัง ZTD timestamps อัตโนมัติ
    Returns: DataFrame [ZTD_m, ZHD_m, ZWD_m, Tm_K, Pi, PWV_mm,
                        dPWV_dt_per15min, surge_flag]
    """
    # Interpolate ERA5 hourly → ZTD timestamps
    combined_idx = era5_df.index.union(ztd_df.index)
    era5_interp  = (era5_df
                    .reindex(combined_idx)
                    .interpolate(method="time")
                    .reindex(ztd_df.index))

    records = []
    for t, row in ztd_df.iterrows():
        if t in era5_interp.index and not era5_interp.loc[t].isna().any():
            ps = float(era5_interp.loc[t, "Ps_hPa"])
            ts = float(era5_interp.loc[t, "Ts_K"])
        else:
            idx = era5_df.index.get_indexer([t], method="nearest")[0]
            ps  = float(era5_df.iloc[idx]["Ps_hPa"])
            ts  = float(era5_df.iloc[idx]["Ts_K"])

        pwv_dict = compute_pwv(
            ztd_m  = float(row["ZTD_m"]),
            Ps_hPa = ps,
            Ts_K   = ts,
        )
        records.append({"time": t, **pwv_dict})

    df = pd.DataFrame(records).set_index("time").sort_index()

    # dPWV/dt (mm / 15 min)
    dt_min = (df.index.to_series()
                .diff()
                .dt.total_seconds()
                .div(60)
                .fillna(5.0))
    df["dPWV_dt_per15min"] = df["PWV_mm"].diff() / dt_min * 15.0

    # Surge flag: |dPWV/dt| > 2 mm / 15 min
    df["surge_flag"] = df["dPWV_dt_per15min"].abs() > 2.0

    log.info(
        f"PWV: {df.PWV_mm.min():.1f}–{df.PWV_mm.max():.1f} mm | "
        f"surge epochs: {int(df.surge_flag.sum())}"
    )
    return df


# ══════════════════════════════════════════════════════════════
# STEP 7 — PLOT
# ══════════════════════════════════════════════════════════════

def plot_pwv(pwv_df: pd.DataFrame, alert_threshold: float = 62.0) -> str:
    """
    2-panel figure บันทึกที่ FIG_DIR:
      Panel 1 : PWV time series + alert threshold + surge highlights
      Panel 2 : dPWV/dt bar chart + ±2 mm/15min threshold lines
    Returns: path ของ saved PNG
    """
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(12, 7), sharex=True,
        gridspec_kw={"height_ratios": [2, 1]},
    )
    fig.suptitle(
        f"GNSS PWV — CUSV Bangkok  |  DOY {OBS_DOY}, {OBS_YEAR}",
        fontsize=14, fontweight="bold",
    )

    # Panel 1: PWV ─────────────────────────────────────────────
    ax1.plot(pwv_df.index, pwv_df["PWV_mm"],
             color="#1a5fa8", linewidth=1.8, label="PWV (mm)", zorder=3)
    ax1.axhline(alert_threshold, color="#e24b4a", linewidth=1.2,
                linestyle="--",
                label=f"Alert threshold ({alert_threshold} mm)", zorder=2)

    surge_epochs = pwv_df[pwv_df["surge_flag"]].index
    for t in surge_epochs:
        ax1.axvspan(t - pd.Timedelta("7.5min"),
                    t + pd.Timedelta("7.5min"),
                    alpha=0.18, color="#ef9f27", zorder=1)
    if not surge_epochs.empty:
        import matplotlib.patches as mpatches
        ax1.add_patch(mpatches.Patch(
            color="#ef9f27", alpha=0.4, label="Surge zone"))

    ax1.set_ylabel("PWV (mm)", fontsize=11)
    ax1.set_ylim(bottom=max(0.0, pwv_df["PWV_mm"].min() - 5))
    ax1.legend(fontsize=9, loc="upper left")
    ax1.grid(axis="y", color="#e0e0e0", linewidth=0.5)
    ax1.spines[["top", "right"]].set_visible(False)

    # Panel 2: dPWV/dt ─────────────────────────────────────────
    bar_colors = ["#e24b4a" if s else "#4a90d9"
                  for s in pwv_df["surge_flag"]]
    ax2.bar(pwv_df.index, pwv_df["dPWV_dt_per15min"],
            width=pd.Timedelta("4min"), color=bar_colors, alpha=0.85)
    ax2.axhline( 2.0, color="#e24b4a", linewidth=0.8, linestyle=":")
    ax2.axhline(-2.0, color="#e24b4a", linewidth=0.8, linestyle=":")
    ax2.axhline( 0.0, color="#888888", linewidth=0.5)
    ax2.set_ylabel("dPWV/dt\n(mm / 15 min)", fontsize=10)
    ax2.set_xlabel("Time (UTC)", fontsize=11)
    ax2.grid(axis="y", color="#e0e0e0", linewidth=0.5)
    ax2.spines[["top", "right"]].set_visible(False)

    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax2.xaxis.set_major_locator(mdates.MinuteLocator(byminute=[0, 15, 30, 45]))
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha="right")

    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, f"pwv_{STATION}_doy{OBS_DOY}_{OBS_YEAR}.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"✅ Figure saved → {out_path}")
    return out_path


# ══════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 62)
    print(f"  GNSS → PWV Pipeline  |  {STATION.upper()} Bangkok")
    print(f"  DOY {OBS_DOY}, {OBS_YEAR}  |  Hour {OBS_HOUR:02d} UTC")
    print(f"  RAW_DIR  : {RAW_DIR}")
    print(f"  PROC_DIR : {PROC_DIR}  ← SSD")
    print(f"  EPH_DIR  : {EPH_DIR}")
    print(f"  ZTD_DIR  : {ZTD_DIR}")
    print(f"  ERA5_DIR : {ERA5_DIR}")
    print(f"  FIG_DIR  : {FIG_DIR}")
    print("=" * 62)

    # STEP 1 ── Decompress + merge ──────────────────────────────
    print("\n[1/6] Decompressing & merging RINEX blocks...")
    rinex_files = decompress_all()
    obs_ds      = merge_rinex_blocks(rinex_files)

    # STEP 2 ── Quality check ───────────────────────────────────
    print("\n[2/6] Quality check...")
    qc = quality_check(obs_ds)

    # STEP 3 ── Precise ephemeris ────────────────────────────────
    print("\n[3/6] Downloading SP3 + CLK...")
    eph = download_ephemeris()

    # STEP 4 ── ZTD ──────────────────────────────────────────────
    print("\n[4/6] PPP → ZTD...")

    if USE_DEMO_ZTD:
        log.info("ใช้ DEMO ZTD (Bangkok Aug: ~2.48 m)  ←  เปลี่ยน USE_DEMO_ZTD=False เพื่อรัน RTKLIB จริง")
        demo_times = pd.date_range(
            f"2024-08-15 {OBS_HOUR:02d}:00",
            periods=12, freq="5min",
        )
        rng    = np.random.default_rng(42)
        ztd_df = pd.DataFrame(
            {"ZTD_m": 2.48 + 0.008 * rng.standard_normal(12)},
            index=demo_times,
        )
    else:
        nav_files = (
            glob.glob(os.path.join(PROC_DIR, f"*.{str(OBS_YEAR)[2:]}p"))
            or glob.glob(os.path.join(PROC_DIR, "*.nav"))
        )
        pos_file = run_ppp(
            rinex_obs = rinex_files[0],
            rinex_nav = nav_files[0] if nav_files else "",
            sp3_path  = eph.get("sp3") or "",
            clk_path  = eph.get("clk") or "",
        )
        ztd_df = parse_ztd(pos_file)

    # STEP 5 ── ERA5 ─────────────────────────────────────────────
    print("\n[5/6] ERA5 surface Ps + Ts...")

    if USE_DEMO_ERA5:
        log.info("ใช้ DEMO ERA5 (Bangkok Aug: ~1007 hPa, ~302 K)  ←  เปลี่ยน USE_DEMO_ERA5=False เพื่อดาวน์โหลดจริง")
        rng2    = np.random.default_rng(99)
        e5_idx  = pd.date_range("2024-08-15 00:00", periods=24, freq="1h")
        era5_df = pd.DataFrame({
            "Ps_hPa": 1007.5 + 0.4 * rng2.standard_normal(24),
            "Ts_K":   302.0  + 0.8 * rng2.standard_normal(24),
        }, index=e5_idx)
    else:
        nc_path = download_era5()
        era5_df = extract_era5_surface(nc_path)

    # STEP 6 ── Compute PWV ──────────────────────────────────────
    print("\n[6/6] Computing ZHD → ZWD → PWV...")
    pwv_df = compute_pwv_timeseries(ztd_df, era5_df)

    print("\n── PWV Results ──")
    print(pwv_df[["ZTD_m", "ZHD_m", "ZWD_m",
                  "PWV_mm", "dPWV_dt_per15min", "surge_flag"]].to_string())

    # STEP 7 ── Plot ─────────────────────────────────────────────
    fig_path = plot_pwv(pwv_df)

    print("\n" + "=" * 62)
    print("✅ Pipeline เสร็จสมบูรณ์")
    print(f"   RINEX  → {PROC_DIR}")
    print(f"   ZTD    → {ZTD_DIR}")
    print(f"   Figure → {fig_path}")
    print("=" * 62)

INFO: พบ 4 ไฟล์ .gz ใน /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw
ERROR:   gunzip ล้มเหลว: cusv2281200.24o
  gunzip: /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw/cusv2281200.24o.gz: not in gzip format
ERROR:   gunzip ล้มเหลว: cusv2281215.24o
  gunzip: /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw/cusv2281215.24o.gz: not in gzip format


  GNSS → PWV Pipeline  |  CUSV Bangkok
  DOY 228, 2024  |  Hour 12 UTC
  RAW_DIR  : /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw
  PROC_DIR : /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_proc  ← SSD
  EPH_DIR  : /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/ephemeris
  ZTD_DIR  : /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/ztd_output
  ERA5_DIR : /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/era5
  FIG_DIR  : /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/figures

[1/6] Decompressing & merging RINEX blocks...


ERROR:   gunzip ล้มเหลว: cusv2281230.24o
  gunzip: /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw/cusv2281230.24o.gz: not in gzip format
ERROR:   gunzip ล้มเหลว: cusv2281245.24o
  gunzip: /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw/cusv2281245.24o.gz: not in gzip format


RuntimeError: ไม่มีไฟล์ RINEX ที่แตกได้เลย — ตรวจสอบไฟล์ .gz ใหม่

In [6]:
"""
GNSS → PWV Pipeline  (CUSV Bangkok)
=====================================
Step 0  : Config & paths
Step 1  : Decompress .gz → RINEX + merge blocks
Step 2  : Quality check ด้วย georinex
Step 3  : ดาวน์โหลด SP3 + CLK (precise ephemeris)
Step 4  : PPP processing ด้วย RTKLIB → ZTD
Step 5  : ดาวน์โหลด ERA5 surface Ps, Ts
Step 6  : คำนวณ ZHD → ZWD → Tm → PWV
Step 7  : Plot PWV time series + dPWV/dt

Install:
    pip install georinex cdsapi xarray numpy pandas matplotlib
    brew install rtklib        # macOS
"""

import os, glob, subprocess, math, logging
import numpy as np
import pandas as pd
import xarray as xr
import georinex as gr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger(__name__)


# ══════════════════════════════════════════════════════════════
# STEP 0 — CONFIG  (แก้ไขเฉพาะส่วนนี้)
# ══════════════════════════════════════════════════════════════

# BASE_DIR = "/Users/toyzaa/Desktop/th_rain"
RAW_DIR  = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw"     # ไฟล์ .gz ที่ดาวน์โหลดมา
PROC_DIR = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_proc"    # ★ RINEX หลัง decompress (Step 1 output / Step 2+ input)
EPH_DIR  = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/ephemeris"    # SP3 + CLK files
ZTD_DIR  = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/ztd_output"   # RTKLIB .pos output
ERA5_DIR = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/era5"         # ERA5 NetCDF
FIG_DIR  = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/figures"      # PNG output

# ── ตรวจสอบ SSD mount ก่อนดำเนินการ ─────────────────────────
_ssd_root = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS"
if not os.path.exists(_ssd_root):
    raise RuntimeError(
        f"ไม่พบ SSD ที่ mount อยู่:\n  {_ssd_root}\n"
        "ตรวจสอบว่าเสียบ drive และ mount แล้ว"
    )

# สร้าง folder อัตโนมัติถ้ายังไม่มี
for _d in [RAW_DIR, PROC_DIR, EPH_DIR, ZTD_DIR, ERA5_DIR, FIG_DIR]:
    os.makedirs(_d, exist_ok=True)

# Station metadata — CUSV Chulalongkorn University, Bangkok
STATION = "cusv"
LAT_DEG = 13.7308    # latitude  (°N)
LON_DEG = 100.5322   # longitude (°E)
H_M     = 35.0       # ellipsoidal height (m)

# Observation date ← ตรงกับไฟล์ cusv2281200.24o.gz
OBS_YEAR = 2024
OBS_DOY  = 228       # day-of-year  (228 = 15 Aug 2024)
OBS_HOUR = 12        # UTC hour ที่เริ่มต้น

# ── Demo flags (เปลี่ยนเป็น False เมื่อมี tool จริง) ──────────
USE_DEMO_ZTD  = True   # True = ใช้ ZTD จำลอง  | False = รัน RTKLIB จริง
USE_DEMO_ERA5 = True   # True = ใช้ ERA5 จำลอง | False = ดาวน์โหลดจาก CDS API


# ══════════════════════════════════════════════════════════════
# STEP 1 — DECOMPRESS .gz  →  RINEX  +  MERGE BLOCKS
# ══════════════════════════════════════════════════════════════

def decompress_all() -> list[str]:
    """
    แตกไฟล์ทุก .gz ใน RAW_DIR → PROC_DIR
    รองรับทั้ง RINEX 2 (.24o) และ RINEX 3 (.rnx)
    Returns: sorted list ของ RINEX paths ที่พร้อมใช้งาน
    """
    gz_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.gz")))
    if not gz_files:
        raise FileNotFoundError(
            f"ไม่พบไฟล์ .gz ใน:\n  {RAW_DIR}\n"
            "ตรวจสอบว่าดาวน์โหลดไฟล์มาไว้ที่ path นี้แล้ว"
        )

    log.info(f"พบ {len(gz_files)} ไฟล์ .gz ใน {RAW_DIR}")
    rinex_paths = []

    for gz in gz_files:
        basename = os.path.basename(gz).replace(".gz", "")
        out_path = os.path.join(PROC_DIR, basename)

        if os.path.exists(out_path):
            log.info(f"  ข้าม (มีอยู่แล้ว): {basename}")
        else:
            with open(out_path, "wb") as out_f:
                result = subprocess.run(
                    ["gunzip", "-c", gz],
                    stdout=out_f,
                    stderr=subprocess.PIPE,
                )
            if result.returncode != 0:
                log.error(
                    f"  gunzip ล้มเหลว: {basename}\n"
                    f"  {result.stderr.decode().strip()}"
                )
                if os.path.exists(out_path):
                    os.remove(out_path)
                continue
            log.info(f"  ✅ แตกไฟล์สำเร็จ: {basename}  →  {PROC_DIR}")

        rinex_paths.append(out_path)

    if not rinex_paths:
        raise RuntimeError("ไม่มีไฟล์ RINEX ที่แตกได้เลย — ตรวจสอบไฟล์ .gz ใหม่")

    log.info(f"รวม {len(rinex_paths)} RINEX blocks พร้อมใช้งานที่ {PROC_DIR}")
    return rinex_paths


def merge_rinex_blocks(rinex_paths: list[str]) -> xr.Dataset:
    """
    โหลดและรวม RINEX blocks ทั้งหมดจาก PROC_DIR (SSD) เป็น xarray.Dataset เดียว
    4 blocks × 15 นาที = 1 ชั่วโมงของข้อมูล 1 Hz (3,600 epochs)
    อ่านจาก: /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_proc
    Returns: xr.Dataset sorted by time
    """
    log.info(f"กำลังโหลด {len(rinex_paths)} blocks จาก SSD:")
    log.info(f"  {PROC_DIR}")
    datasets = []

    for path in rinex_paths:
        try:
            ds = gr.load(path, use=["C1C", "L1C", "L2C", "S1C", "S2C"])
            datasets.append(ds)
            log.info(
                f"  {os.path.basename(path)} | "
                f"epochs={len(ds.time):,} | "
                f"sats={len(ds.sv)}"
            )
        except Exception as e:
            log.warning(f"  อ่านไม่ได้: {os.path.basename(path)} — {e}")

    if not datasets:
        raise RuntimeError("ไม่สามารถอ่าน RINEX block ใดได้เลย")

    combined = xr.concat(datasets, dim="time").sortby("time")
    t0 = str(combined.time.values[0])[:19]
    t1 = str(combined.time.values[-1])[:19]
    log.info(f"Merged: {len(combined.time):,} epochs | {t0} → {t1}")
    return combined


# ══════════════════════════════════════════════════════════════
# STEP 2 — QUALITY CHECK
# ══════════════════════════════════════════════════════════════

def quality_check(obs_ds: xr.Dataset) -> dict:
    """
    ตรวจคุณภาพ RINEX dataset ที่อ่านมาจาก SSD
      path: /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_proc
    ตรวจ: SNR, จำนวน satellites, completeness, multipath proxy
    Returns: dict สรุป QC metrics
    """
    qc = {
        "n_epochs" : int(len(obs_ds.time)),
        "n_sats"   : int(len(obs_ds.sv)),
        "t_start"  : str(obs_ds.time.values[0])[:19],
        "t_end"    : str(obs_ds.time.values[-1])[:19],
    }

    # ── SNR L1 (S1C) ──────────────────────────────────────────
    if "S1C" in obs_ds:
        snr = obs_ds["S1C"].values.flatten()
        snr = snr[~np.isnan(snr)]
        qc["snr_mean_dBHz"]   = round(float(snr.mean()), 1)
        qc["snr_min_dBHz"]    = round(float(snr.min()),  1)
        qc["snr_pct_below30"] = round(float((snr < 30).mean() * 100), 1)
        status = "⚠️  ต่ำ" if qc["snr_mean_dBHz"] < 30 else "✅ ดี"
        log.info(f"  SNR L1: {qc['snr_mean_dBHz']} dBHz mean  {status}")

    # ── Multipath proxy ────────────────────────────────────────
    if "C1C" in obs_ds and "L1C" in obs_ds:
        mp = (obs_ds["C1C"] - obs_ds["L1C"]).values.flatten()
        mp = mp[~np.isnan(mp)]
        if len(mp) > 0:
            qc["multipath_proxy_m"] = round(float(np.std(mp)), 3)
            log.info(f"  Multipath proxy σ: {qc['multipath_proxy_m']} m")

    # ── Data completeness (epochs ที่มี ≥ 4 sats) ─────────────
    if "C1C" in obs_ds:
        n_valid = (~np.isnan(obs_ds["C1C"].values)).sum(axis=1)
        pct     = float((n_valid >= 4).mean() * 100)
        qc["completeness_pct"] = round(pct, 1)
        log.info(f"  Completeness (≥4 sats): {pct:.1f}%")

    log.info("── QC Summary ──")
    for k, v in qc.items():
        log.info(f"  {k}: {v}")
    return qc


# ══════════════════════════════════════════════════════════════
# STEP 3 — ดาวน์โหลด SP3 + CLK (Precise Ephemeris)
# ══════════════════════════════════════════════════════════════

def doy_to_gpsweek(year: int, doy: int) -> tuple[int, int]:
    """แปลง year + doy → (GPS week, day-of-week)"""
    from datetime import date, timedelta
    d          = date(year, 1, 1) + timedelta(days=doy - 1)
    gps_epoch  = date(1980, 1, 6)
    total_days = (d - gps_epoch).days
    return total_days // 7, total_days % 7


def download_ephemeris() -> dict[str, str | None]:
    """
    ดาวน์โหลด IGS Final SP3 + CLK จาก NASA CDDIS → EPH_DIR
    หมายเหตุ: IGS Final products พร้อมใช้หลัง ~13 วัน
    Returns: {"sp3": path_or_None, "clk": path_or_None}
    """
    import urllib.request

    gps_week, gps_dow = doy_to_gpsweek(OBS_YEAR, OBS_DOY)
    log.info(f"GPS week={gps_week}, dow={gps_dow}  →  EPH_DIR: {EPH_DIR}")

    base  = "https://cddis.nasa.gov/archive/gnss/products"
    files = {
        "sp3": f"igs{gps_week}{gps_dow}.sp3.Z",
        "clk": f"igs{gps_week}{gps_dow}.clk_30s.Z",
    }

    paths: dict[str, str | None] = {}
    for ftype, fname in files.items():
        final_path = os.path.join(EPH_DIR, fname.replace(".Z", ""))

        if os.path.exists(final_path):
            log.info(f"  มีอยู่แล้ว: {os.path.basename(final_path)}")
            paths[ftype] = final_path
            continue

        url      = f"{base}/{gps_week}/{fname}"
        tmp_path = os.path.join(EPH_DIR, fname)

        log.info(f"  ดาวน์โหลด {ftype.upper()}: {url}")
        try:
            urllib.request.urlretrieve(url, tmp_path)
            subprocess.run(["uncompress", tmp_path], check=True)
            paths[ftype] = final_path
            log.info(f"  ✅ {ftype.upper()} พร้อมที่: {final_path}")
        except Exception as e:
            log.error(f"  ❌ ล้มเหลว: {fname} — {e}")
            paths[ftype] = None

    return paths


# ══════════════════════════════════════════════════════════════
# STEP 4 — PPP PROCESSING (RTKLIB) → ZTD time series
# ══════════════════════════════════════════════════════════════

RTKLIB_CONF = """\
pos1-posmode    =ppp-static
pos1-frequency  =l1+l2
pos1-tropopt    =est-ztd
pos1-ionoopt    =dual-freq
pos1-sateph     =prec
pos1-navsys     =1+2+4+8
pos1-elmask     =15
pos2-armode     =off
out-outstat     =residual
out-solstat     =0
"""


def run_ppp(rinex_obs: str, rinex_nav: str,
            sp3_path: str, clk_path: str) -> str:
    """
    รัน RTKLIB rnx2rtkp ใน PPP-static mode → ZTD
    Output .pos file บันทึกที่ ZTD_DIR
    Returns: path ของ .pos output file
    """
    conf_path = os.path.join(ZTD_DIR, "ppp.conf")
    out_path  = os.path.join(ZTD_DIR, f"{STATION}_ztd.pos")

    with open(conf_path, "w") as f:
        f.write(RTKLIB_CONF)

    for label, path in [("obs", rinex_obs), ("nav", rinex_nav),
                         ("sp3", sp3_path),  ("clk", clk_path)]:
        if not path or not os.path.exists(path):
            raise FileNotFoundError(f"ไม่พบ {label} file: {path}")

    cmd = ["rnx2rtkp", "-k", conf_path, "-x", "4",
           "-o", out_path,
           rinex_obs, rinex_nav, sp3_path, clk_path]

    log.info(f"รัน RTKLIB PPP:\n  {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        log.error(f"RTKLIB error:\n{result.stderr[-800:]}")
    else:
        log.info(f"✅ PPP เสร็จ → {out_path}")
    return out_path


def parse_ztd(pos_file: str) -> pd.DataFrame:
    """
    อ่าน ZTD จาก RTKLIB .pos output (debug level 4)
    Column order: date time lat lon h Q ns sdn sde sdu sdne sdeu sdun age ZTD
    Returns: DataFrame index=time, columns=[ZTD_m]
    """
    if not os.path.exists(pos_file):
        raise FileNotFoundError(f"ไม่พบ pos file: {pos_file}")

    rows = []
    with open(pos_file) as f:
        for line in f:
            if line.startswith("%") or not line.strip():
                continue
            parts = line.split()
            if len(parts) < 15:
                continue
            try:
                epoch = pd.Timestamp(f"{parts[0]} {parts[1]}")
                ztd   = float(parts[14])    # ZTD (m) — column 14
                rows.append({"time": epoch, "ZTD_m": ztd})
            except (ValueError, IndexError):
                continue

    if not rows:
        raise RuntimeError(f"ไม่พบ ZTD data ใน {pos_file}")

    df = pd.DataFrame(rows).set_index("time").sort_index().dropna()
    log.info(f"ZTD: {len(df)} epochs | {df.ZTD_m.min():.4f}–{df.ZTD_m.max():.4f} m")
    return df


# ══════════════════════════════════════════════════════════════
# STEP 5 — ERA5 Surface Ps, Ts
# ══════════════════════════════════════════════════════════════

def download_era5() -> str:
    """
    ดาวน์โหลด ERA5 single-level (surface_pressure + 2m_temperature) ทั้งวัน
    บันทึกที่ ERA5_DIR
    ต้องตั้ง ~/.cdsapirc ก่อน: https://cds.climate.copernicus.eu/api-how-to
    Returns: path ของ NetCDF file
    """
    import cdsapi
    from datetime import date, timedelta

    d       = date(OBS_YEAR, 1, 1) + timedelta(days=OBS_DOY - 1)
    datestr = d.strftime("%Y-%m-%d")
    outpath = os.path.join(ERA5_DIR, f"era5_sfc_{datestr}.nc")

    if os.path.exists(outpath):
        log.info(f"ERA5 มีอยู่แล้วที่: {outpath}")
        return outpath

    area = [LAT_DEG + 1, LON_DEG - 1, LAT_DEG - 1, LON_DEG + 1]   # [N,W,S,E]
    log.info(f"ดาวน์โหลด ERA5: {datestr}  area={area}  →  {ERA5_DIR}")

    c = cdsapi.Client()
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable":     ["surface_pressure", "2m_temperature"],
            "year":         str(OBS_YEAR),
            "month":        f"{d.month:02d}",
            "day":          f"{d.day:02d}",
            "time":         [f"{h:02d}:00" for h in range(24)],
            "area":         area,
            "format":       "netcdf",
        },
        outpath,
    )
    log.info(f"✅ ERA5 ดาวน์โหลดเสร็จ → {outpath}")
    return outpath


def extract_era5_surface(nc_path: str) -> pd.DataFrame:
    """
    Extract Ps (hPa) และ Ts (K) ที่จุดใกล้ station ที่สุดจาก ERA5 NetCDF
    Returns: DataFrame index=time, columns=[Ps_hPa, Ts_K]
    """
    ds     = xr.open_dataset(nc_path)
    pt     = ds.sel(latitude=LAT_DEG, longitude=LON_DEG, method="nearest")
    Ps_hPa = pt["sp"].values  / 100.0    # Pa → hPa
    Ts_K   = pt["t2m"].values            # K
    times  = pd.to_datetime(pt["time"].values)

    df = pd.DataFrame({"Ps_hPa": Ps_hPa, "Ts_K": Ts_K}, index=times)
    log.info(
        f"ERA5 | Ps={Ps_hPa.mean():.1f}±{Ps_hPa.std():.1f} hPa | "
        f"Ts={Ts_K.mean():.1f}±{Ts_K.std():.1f} K"
    )
    return df


# ══════════════════════════════════════════════════════════════
# STEP 6 — ZTD → ZHD → ZWD → Tm → Pi → PWV
# ══════════════════════════════════════════════════════════════

def compute_pwv(ztd_m: float, Ps_hPa: float, Ts_K: float) -> dict:
    """
    คำนวณ PWV จาก ZTD + surface meteorological parameters

    Chain:
        ZHD  = 0.0022768 × Ps / f(φ, H)     [Saastamoinen 1972]
        ZWD  = ZTD − ZHD
        Tm   = 70.2 + 0.72 × Ts             [Bevis et al. 1992]
        Π    = 1e6 / [Rv × (k3/Tm + k2')]
        PWV  = Π × ZWD  (mm)

    Constants [Bevis 1994]:
        k2'  = 22.97   K/hPa
        k3   = 377600  K²/hPa
        Rv   = 461.495 J/(kg·K)
    """
    # 1. ZHD (Saastamoinen 1972)
    phi   = math.radians(LAT_DEG)
    f_lat = 1.0 - 0.00266 * math.cos(2.0 * phi) - 0.00028e-3 * H_M
    ZHD   = 0.0022768 * Ps_hPa / f_lat           # metres

    # 2. ZWD
    ZWD = ztd_m - ZHD                             # metres
    if ZWD < 0:
        log.warning(
            f"ZWD < 0 ({ZWD:.5f} m) — "
            f"ตรวจสอบ Ps={Ps_hPa:.1f} hPa / ZTD={ztd_m:.4f} m"
        )

    # 3. Weighted mean temperature (Bevis 1992 linear approx)
    Tm = 70.2 + 0.72 * Ts_K                       # K

    # 4. Pi conversion factor
    Rv  = 461.495    # J/(kg·K)
    k2p = 22.97      # K/hPa
    k3  = 377600.0   # K²/hPa
    Pi  = 1e6 / (Rv * (k3 / Tm + k2p))

    # 5. PWV
    PWV = Pi * ZWD * 1000.0                        # m → mm

    return {
        "ZHD_m":  round(ZHD,  5),
        "ZWD_m":  round(ZWD,  5),
        "Tm_K":   round(Tm,   2),
        "Pi":     round(Pi,   5),
        "PWV_mm": round(PWV,  2),
    }


def compute_pwv_timeseries(ztd_df: pd.DataFrame,
                            era5_df: pd.DataFrame) -> pd.DataFrame:
    """
    คำนวณ PWV ทุก epoch + dPWV/dt (rainbomb precursor indicator)
    ERA5 (hourly) จะถูก interpolate ไปยัง ZTD timestamps อัตโนมัติ
    Returns: DataFrame [ZTD_m, ZHD_m, ZWD_m, Tm_K, Pi, PWV_mm,
                        dPWV_dt_per15min, surge_flag]
    """
    # Interpolate ERA5 hourly → ZTD timestamps
    combined_idx = era5_df.index.union(ztd_df.index)
    era5_interp  = (era5_df
                    .reindex(combined_idx)
                    .interpolate(method="time")
                    .reindex(ztd_df.index))

    records = []
    for t, row in ztd_df.iterrows():
        if t in era5_interp.index and not era5_interp.loc[t].isna().any():
            ps = float(era5_interp.loc[t, "Ps_hPa"])
            ts = float(era5_interp.loc[t, "Ts_K"])
        else:
            idx = era5_df.index.get_indexer([t], method="nearest")[0]
            ps  = float(era5_df.iloc[idx]["Ps_hPa"])
            ts  = float(era5_df.iloc[idx]["Ts_K"])

        pwv_dict = compute_pwv(
            ztd_m  = float(row["ZTD_m"]),
            Ps_hPa = ps,
            Ts_K   = ts,
        )
        records.append({"time": t, **pwv_dict})

    df = pd.DataFrame(records).set_index("time").sort_index()

    # dPWV/dt (mm / 15 min)
    dt_min = (df.index.to_series()
                .diff()
                .dt.total_seconds()
                .div(60)
                .fillna(5.0))
    df["dPWV_dt_per15min"] = df["PWV_mm"].diff() / dt_min * 15.0

    # Surge flag: |dPWV/dt| > 2 mm / 15 min
    df["surge_flag"] = df["dPWV_dt_per15min"].abs() > 2.0

    log.info(
        f"PWV: {df.PWV_mm.min():.1f}–{df.PWV_mm.max():.1f} mm | "
        f"surge epochs: {int(df.surge_flag.sum())}"
    )
    return df


# ══════════════════════════════════════════════════════════════
# STEP 7 — PLOT
# ══════════════════════════════════════════════════════════════

def plot_pwv(pwv_df: pd.DataFrame, alert_threshold: float = 62.0) -> str:
    """
    2-panel figure บันทึกที่ FIG_DIR:
      Panel 1 : PWV time series + alert threshold + surge highlights
      Panel 2 : dPWV/dt bar chart + ±2 mm/15min threshold lines
    Returns: path ของ saved PNG
    """
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(12, 7), sharex=True,
        gridspec_kw={"height_ratios": [2, 1]},
    )
    fig.suptitle(
        f"GNSS PWV — CUSV Bangkok  |  DOY {OBS_DOY}, {OBS_YEAR}",
        fontsize=14, fontweight="bold",
    )

    # Panel 1: PWV ─────────────────────────────────────────────
    ax1.plot(pwv_df.index, pwv_df["PWV_mm"],
             color="#1a5fa8", linewidth=1.8, label="PWV (mm)", zorder=3)
    ax1.axhline(alert_threshold, color="#e24b4a", linewidth=1.2,
                linestyle="--",
                label=f"Alert threshold ({alert_threshold} mm)", zorder=2)

    surge_epochs = pwv_df[pwv_df["surge_flag"]].index
    for t in surge_epochs:
        ax1.axvspan(t - pd.Timedelta("7.5min"),
                    t + pd.Timedelta("7.5min"),
                    alpha=0.18, color="#ef9f27", zorder=1)
    if not surge_epochs.empty:
        import matplotlib.patches as mpatches
        ax1.add_patch(mpatches.Patch(
            color="#ef9f27", alpha=0.4, label="Surge zone"))

    ax1.set_ylabel("PWV (mm)", fontsize=11)
    ax1.set_ylim(bottom=max(0.0, pwv_df["PWV_mm"].min() - 5))
    ax1.legend(fontsize=9, loc="upper left")
    ax1.grid(axis="y", color="#e0e0e0", linewidth=0.5)
    ax1.spines[["top", "right"]].set_visible(False)

    # Panel 2: dPWV/dt ─────────────────────────────────────────
    bar_colors = ["#e24b4a" if s else "#4a90d9"
                  for s in pwv_df["surge_flag"]]
    ax2.bar(pwv_df.index, pwv_df["dPWV_dt_per15min"],
            width=pd.Timedelta("4min"), color=bar_colors, alpha=0.85)
    ax2.axhline( 2.0, color="#e24b4a", linewidth=0.8, linestyle=":")
    ax2.axhline(-2.0, color="#e24b4a", linewidth=0.8, linestyle=":")
    ax2.axhline( 0.0, color="#888888", linewidth=0.5)
    ax2.set_ylabel("dPWV/dt\n(mm / 15 min)", fontsize=10)
    ax2.set_xlabel("Time (UTC)", fontsize=11)
    ax2.grid(axis="y", color="#e0e0e0", linewidth=0.5)
    ax2.spines[["top", "right"]].set_visible(False)

    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax2.xaxis.set_major_locator(mdates.MinuteLocator(byminute=[0, 15, 30, 45]))
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha="right")

    plt.tight_layout()
    out_path = os.path.join(FIG_DIR, f"pwv_{STATION}_doy{OBS_DOY}_{OBS_YEAR}.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"✅ Figure saved → {out_path}")
    return out_path


# ══════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("=" * 62)
    print(f"  GNSS → PWV Pipeline  |  {STATION.upper()} Bangkok")
    print(f"  DOY {OBS_DOY}, {OBS_YEAR}  |  Hour {OBS_HOUR:02d} UTC")
    print(f"  RAW_DIR  : {RAW_DIR}")
    print(f"  PROC_DIR : {PROC_DIR}  ← SSD")
    print(f"  EPH_DIR  : {EPH_DIR}")
    print(f"  ZTD_DIR  : {ZTD_DIR}")
    print(f"  ERA5_DIR : {ERA5_DIR}")
    print(f"  FIG_DIR  : {FIG_DIR}")
    print("=" * 62)

    # STEP 1 ── Decompress + merge ──────────────────────────────
    print("\n[1/6] Decompressing & merging RINEX blocks...")
    rinex_files = decompress_all()
    obs_ds      = merge_rinex_blocks(rinex_files)

    # STEP 2 ── Quality check ───────────────────────────────────
    print("\n[2/6] Quality check...")
    qc = quality_check(obs_ds)

    # STEP 3 ── Precise ephemeris ────────────────────────────────
    print("\n[3/6] Downloading SP3 + CLK...")
    eph = download_ephemeris()

    # STEP 4 ── ZTD ──────────────────────────────────────────────
    print("\n[4/6] PPP → ZTD...")

    if USE_DEMO_ZTD:
        log.info("ใช้ DEMO ZTD (Bangkok Aug: ~2.48 m)  ←  เปลี่ยน USE_DEMO_ZTD=False เพื่อรัน RTKLIB จริง")
        demo_times = pd.date_range(
            f"2024-08-15 {OBS_HOUR:02d}:00",
            periods=12, freq="5min",
        )
        rng    = np.random.default_rng(42)
        ztd_df = pd.DataFrame(
            {"ZTD_m": 2.48 + 0.008 * rng.standard_normal(12)},
            index=demo_times,
        )
    else:
        nav_files = (
            glob.glob(os.path.join(PROC_DIR, f"*.{str(OBS_YEAR)[2:]}p"))
            or glob.glob(os.path.join(PROC_DIR, "*.nav"))
        )
        pos_file = run_ppp(
            rinex_obs = rinex_files[0],
            rinex_nav = nav_files[0] if nav_files else "",
            sp3_path  = eph.get("sp3") or "",
            clk_path  = eph.get("clk") or "",
        )
        ztd_df = parse_ztd(pos_file)

    # STEP 5 ── ERA5 ─────────────────────────────────────────────
    print("\n[5/6] ERA5 surface Ps + Ts...")

    if USE_DEMO_ERA5:
        log.info("ใช้ DEMO ERA5 (Bangkok Aug: ~1007 hPa, ~302 K)  ←  เปลี่ยน USE_DEMO_ERA5=False เพื่อดาวน์โหลดจริง")
        rng2    = np.random.default_rng(99)
        e5_idx  = pd.date_range("2024-08-15 00:00", periods=24, freq="1h")
        era5_df = pd.DataFrame({
            "Ps_hPa": 1007.5 + 0.4 * rng2.standard_normal(24),
            "Ts_K":   302.0  + 0.8 * rng2.standard_normal(24),
        }, index=e5_idx)
    else:
        nc_path = download_era5()
        era5_df = extract_era5_surface(nc_path)

    # STEP 6 ── Compute PWV ──────────────────────────────────────
    print("\n[6/6] Computing ZHD → ZWD → PWV...")
    pwv_df = compute_pwv_timeseries(ztd_df, era5_df)

    print("\n── PWV Results ──")
    print(pwv_df[["ZTD_m", "ZHD_m", "ZWD_m",
                  "PWV_mm", "dPWV_dt_per15min", "surge_flag"]].to_string())

    # STEP 7 ── Plot ─────────────────────────────────────────────
    fig_path = plot_pwv(pwv_df)

    print("\n" + "=" * 62)
    print("✅ Pipeline เสร็จสมบูรณ์")
    print(f"   RINEX  → {PROC_DIR}")
    print(f"   ZTD    → {ZTD_DIR}")
    print(f"   Figure → {fig_path}")
    print("=" * 62)

INFO: พบ 4 ไฟล์ .gz ใน /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw
ERROR:   gunzip ล้มเหลว: cusv2281200.24o
  gunzip: /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw/cusv2281200.24o.gz: not in gzip format


  GNSS → PWV Pipeline  |  CUSV Bangkok
  DOY 228, 2024  |  Hour 12 UTC
  RAW_DIR  : /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw
  PROC_DIR : /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_proc  ← SSD
  EPH_DIR  : /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/ephemeris
  ZTD_DIR  : /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/ztd_output
  ERA5_DIR : /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/era5
  FIG_DIR  : /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/figures

[1/6] Decompressing & merging RINEX blocks...


ERROR:   gunzip ล้มเหลว: cusv2281215.24o
  gunzip: /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw/cusv2281215.24o.gz: not in gzip format
ERROR:   gunzip ล้มเหลว: cusv2281230.24o
  gunzip: /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw/cusv2281230.24o.gz: not in gzip format
ERROR:   gunzip ล้มเหลว: cusv2281245.24o
  gunzip: /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw/cusv2281245.24o.gz: not in gzip format


RuntimeError: ไม่มีไฟล์ RINEX ที่แตกได้เลย — ตรวจสอบไฟล์ .gz ใหม่

In [7]:
import os, glob, subprocess

RAW_DIR  = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw"
PROC_DIR = "/Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_proc"

# ── 1. ตรวจว่ามีไฟล์ .gz จริงไหม ──────────────────────────
gz_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.gz")))
print(f"พบ .gz files: {len(gz_files)}")
for f in gz_files:
    print(f"  {os.path.basename(f)}  ({os.path.getsize(f):,} bytes)")

# ── 2. ตรวจ gunzip ทำงานได้ไหม ─────────────────────────────
print(f"\ngunzip version:")
subprocess.run(["gunzip", "--version"], capture_output=False)

# ── 3. ลองแตกไฟล์แรกด้วย verbose ──────────────────────────
if gz_files:
    test_gz   = gz_files[0]
    test_out  = os.path.join(PROC_DIR, os.path.basename(test_gz).replace(".gz", ""))
    print(f"\nทดสอบแตกไฟล์: {os.path.basename(test_gz)}")
    print(f"  ปลายทาง: {test_out}")

    with open(test_out, "wb") as out_f:
        result = subprocess.run(
            ["gunzip", "-c", test_gz],
            stdout=out_f,
            stderr=subprocess.PIPE,
        )
    print(f"  returncode : {result.returncode}")
    print(f"  stderr     : {result.stderr.decode().strip() or '(ว่าง)'}")

    if os.path.exists(test_out):
        size = os.path.getsize(test_out)
        print(f"  output size: {size:,} bytes  {'✅' if size > 0 else '❌ ไฟล์ว่าง!'}")
    else:
        print("  ❌ ไม่มีไฟล์ output เลย")

พบ .gz files: 4
  cusv2281200.24o.gz  (11,249 bytes)
  cusv2281215.24o.gz  (11,249 bytes)
  cusv2281230.24o.gz  (9,690 bytes)
  cusv2281245.24o.gz  (11,249 bytes)

gunzip version:

ทดสอบแตกไฟล์: cusv2281200.24o.gz
  ปลายทาง: /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_proc/cusv2281200.24o
  returncode : 1
  stderr     : gunzip: /Volumes/InnoGI/SSD/Thanabordee/_data/GNSS/gnss_raw/cusv2281200.24o.gz: not in gzip format
  output size: 0 bytes  ❌ ไฟล์ว่าง!


Apple gzip 479
